# Сборный проект





### Идея решения задачи от заказчика: 

Создать модель предсказания ДТП (целевое значение — at_fault (виновник) в таблице parties)

Для модели выбрать тип виновника — только машина (car).

Выбрать случаи, когда ДТП привело к любым повреждениям транспортного средства, кроме типа SCRATCH (царапина).

Для моделирования ограничиться данными за 2012 год — они самые свежие.

Обязательное условие — учесть фактор возраста автомобиля.

На основе модели исследовать основные факторы ДТП.

Понять, помогут ли результаты моделирования и анализ важности факторов ответить на вопросы:

Возможно ли создать адекватную системы оценки водительского риска при выдаче авто?

Какие ещё факторы нужно учесть?

Нужно ли оборудовать автомобиль какими-либо датчиками или камерой?

Заказчик предлагает вам поработать с базой данных по происшествиям и сформировать свои идеи создания такой системы. 

## Краткое описание таблиц

collisions — общая информация о ДТП

Имеет уникальный case_id. Эта таблица описывает общую информацию о ДТП. Например, где оно произошло и когда.
parties — информация об участниках ДТП

Имеет неуникальный case_id, который сопоставляется с соответствующим ДТП в таблице collisions. Каждая строка здесь описывает одну из сторон, участвующих в ДТП. Если столкнулись две машины, в этой таблице должно быть две строки с совпадением case_id. Если нужен уникальный идентификатор, это case_id and party_number.
vehicles — информация о пострадавших машинах

Имеет неуникальные case_id и неуникальные party_number, которые сопоставляются с таблицей collisions и таблицей parties. Если нужен уникальный идентификатор, это case_id and party_number.

## Установка и импорт необходимых библиотек

In [1]:
!pip install -q phik
!pip install -U scikit-learn

^C
Traceback (most recent call last):
  File "<frozen importlib._bootstrap_external>", line 148, in _path_is_mode_type
  File "<frozen importlib._bootstrap_external>", line 142, in _path_stat
FileNotFoundError: [Errno 2] No such file or directory: '/opt/conda/lib/python3.9/site-packages/pip/_internal/index/__init__.cpython-39-x86_64-linux-gnu.so'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/bin/pip", line 10, in <module>
    sys.exit(main())
  File "/opt/conda/lib/python3.9/site-packages/pip/_internal/cli/main.py", line 69, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/opt/conda/lib/python3.9/site-packages/pip/_internal/commands/__init__.py", line 91, in create_command
    module = importlib.import_module(module_path)
  File "/opt/conda/lib/python3.9/importlib/__init__.py", line 127, in import_module
    return _bootstrap._gcd_import(name[level:], package, level

In [2]:
import phik

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from phik.report import plot_correlation_matrix
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (confusion_matrix, f1_score, recall_score, 
                           roc_auc_score, roc_curve, precision_score, 
                           precision_recall_curve, classification_report)
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, 
                                 StandardScaler)


from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool


In [3]:
RANDOM_STATE = 89

## Подключитесь к базе. Загрузите таблицы sql

Шаг 1. Загрузите таблицы sql
Подключитесь к базе данных, используя данные:


In [4]:
db_config = {
'user': 'praktikum_student', # имя пользователя,
'pwd': 'Sdf4$2;d-d30pp', # пароль,
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # порт подключения,
'db': 'data-science-vehicle-db' # название базы данных,
} 

In [5]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

In [ ]:
engine = create_engine(connection_string)

In [ ]:
query = '''
SELECT *
FROM information_schema.tables
WHERE table_schema='public'
      AND table_type='BASE TABLE';
'''   
display(pd.read_sql_query(query, con=engine))

На основе предоставленного SQL-запроса и вывода, мы можем провести первичное исследование таблиц:

Наличие таблиц и их количество:

В базе данных найдено 4 таблицы: case_ids, collisions, parties и vehicles (table_name)

Это соответствует условию задачи, где ожидались именно эти 4 таблицы. Хорошо.

In [ ]:
# уникальность case_id
query = '''
SELECT case_id, COUNT(*) as count
FROM case_ids
GROUP BY case_id
HAVING COUNT(*) > 1;
'''
display(pd.read_sql_query(query, con=engine))

## Проведите первичное исследование таблиц

https://pictures.s3.yandex.net/resources/PPROD-4019_1725013595.png

Еще раз взглянем на ER - диаграмму. Ореинтировка по ней не будет лишней.

Начнем с главного связующего звена. Оно же - case_ids - самое простое по структуре.

#### case_ids

In [ ]:
query = '''
SELECT *
FROM case_ids
LIMIT 10;
'''

display(pd.read_sql_query(query, con=engine))

In [ ]:
#  распределение по годам(году)
query = '''
SELECT db_year, COUNT(*) as cases_count
FROM case_ids
GROUP BY db_year
ORDER BY db_year;
'''
display(pd.read_sql_query(query, con=engine))

2021. Допустим. Пусть case_ids и состоит из 2-х столбцов, проверить его все же надо. Но пока я  не знаю что делать с этой информацией.

Кроме того, что это год загрузки данных, судя по условию, а значит - выше быть не может

#### collisions

In [ ]:
query = '''
SELECT *
FROM collisions
LIMIT 10;
'''

display(pd.read_sql_query(query, con=engine))

Тут основная информация с подробностями.

In [ ]:
query = '''
SELECT c.*, ci.db_year 
FROM collisions c
JOIN case_ids ci ON c.case_id = ci.case_id
LIMIT 10;
'''
display(pd.read_sql_query(query, con=engine))

query = '''
SELECT type_of_collision, COUNT(*) as count
FROM collisions
GROUP BY type_of_collision
ORDER BY count DESC;
'''

display(pd.read_sql_query(query, con=engine))

query = '''
SELECT EXTRACT(HOUR FROM collision_time) as hour, COUNT(*) as accidents
FROM collisions
GROUP BY hour
ORDER BY hour;
'''
display(pd.read_sql_query(query, con=engine))

Тип аварии	TYPE_OF_COLLISION	A — Head-On (Лоб в лоб)
B — Sideswipe (Сторона)
C — Rear End (Столкновение задней частью)
D — Broadside (Боковой удар)
E — Hit Object (Удар объекта)
F — Overturned (Опрокинутый)
G — Vehicle (транспортное средство/ Пешеход)
H — Other (Другое)
- — Not Stated (Не указано)


rear end - почти треть от общих типов столкновений. Потом посчитаю




Дата происшествия	COLLISION_DATE	Формат год/месяц/день
Время происшествия	COLLISION_TIME	Формат: 24-часовой

17:00 - абсолютное большинство столкновений. 
Ночью 4:00 - минимум. Да и вообще в дневное время- больше всего

    Даты будут проанализированны позже

#### parties

In [ ]:
query = '''
SELECT *
FROM parties
LIMIT 10;
'''

display(pd.read_sql_query(query, con=engine))

In [ ]:
#распределение типов участников

query = '''
SELECT party_type, COUNT(*) as count
FROM parties
GROUP BY party_type
ORDER BY count DESC;
'''

display(pd.read_sql_query(query, con=engine))


# вина
query = '''
SELECT 
    at_fault,
    COUNT(*) as count
FROM parties
GROUP BY at_fault;
'''

display(pd.read_sql_query(query, con=engine))

# алкоголь
query = '''
SELECT 
    party_sobriety,
    COUNT(*) as count
FROM parties
WHERE party_type = 'car'
GROUP BY party_sobriety
ORDER BY count DESC;
'''

display(pd.read_sql_query(query, con=engine))

Машины врезаются в машины. В очень большом и подавляющем большинстве.

Следовательно, если машины врезаются в машины, то и участников ДТП в большинстве случаев - минимум 2. (если эта машина не была припаркована и виновник ДТП в нее не врезался. Является ли припаркованная в неправильном месте машина причиной ДТП?)

Исходя из первого вывода, следует второе: Виновность и невиновность распределены почти что поровну.

Ну и большинство аварий было на трезвую голову. Но в эту статистику входят и невиновные, а потому данные не следует рассматривать как причину ДТП. Это мы исключим в последующих вычислениях, когда отберем год итд

Идея решения задачи от заказчика. Первый и второй пункты:

Создать модель предсказания ДТП (целевое значение — at_fault (виновник) в таблице parties)

Для модели выбрать тип виновника — только машина (car).



In [ ]:
query = '''
SELECT 
    cellphone_in_use,
    COUNT(*) as count
FROM parties
WHERE party_type = 'car'
GROUP BY cellphone_in_use;
'''
display(pd.read_sql_query(query, con=engine))

Пользователей телефона мало

#### vehicles

In [ ]:
query = '''
SELECT *
FROM vehicles
LIMIT 10;
'''

display(pd.read_sql_query(query, con=engine))

In [ ]:
query = '''
SELECT 
    vehicle_type,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM vehicles), 1) as percentage
FROM vehicles
GROUP BY vehicle_type
ORDER BY count DESC;
'''

display(pd.read_sql_query(query, con=engine))



In [ ]:
query = '''
SELECT 
    vehicle_transmission,
    COUNT(*) as count,
    AVG(vehicle_age) as avg_age
FROM vehicles
GROUP BY vehicle_transmission;
'''

display(pd.read_sql_query(query, con=engine))


In [ ]:
#На этот признак заказчиком сделан акцент

query = '''
SELECT 
    CASE
        WHEN vehicle_age <= 2 THEN '0-2 года'
        WHEN vehicle_age <= 5 THEN '3-5 лет'
        WHEN vehicle_age <= 10 THEN '6-10 лет'
        ELSE '10 и больше годов'
    END as age_group,
    COUNT(*) as count
FROM vehicles
GROUP BY age_group
ORDER BY MIN(vehicle_age);
'''
display(pd.read_sql_query(query, con=engine))

Идея решения задачи от заказчика. 

Обязательное условие — учесть фактор возраста автомобиля.


Вывод:

Общий ключ между таблицами - это case_id, который позволяет соединять данные о ДТП (collisions) с информацией об участниках (parties) и их транспортных средствах (vehicles). Я его рассмотрел в начале этого этапа. Он присутствует во всех таблицах.
Вот как таблицы связаны между собой:

collisions & parties: связь через case_id

collisions & vehicles: связь через case_id

parties & vehicles: связь через case_id и party_number

А еще в таблицах parties и vehicles также используется поле party_number для связи между собой.

##  Проведите статистический анализ факторов ДТП

Выясните, в какие месяцы происходит наибольшее количество аварий. 

Проанализируйте весь период наблюдений (таблица collisions).
Создайте sql-запрос;
Постройте график;
Сделайте вывод.


In [ ]:
query = '''
SELECT COUNT(case_id) AS total_ids,
       DATE_TRUNC('month', collision_date)::date AS month
FROM collisions
GROUP BY DATE_TRUNC('month', collision_date)
ORDER BY DATE_TRUNC('month', collision_date);
'''


fig = px.bar(
    pd.read_sql_query(query, con=engine), 
    x='month', 
    y='total_ids',
    title='Статистика происшествий по месяцам',
    labels={'total_ids': 'Количество ДТП', 'month': 'Месяц'},
    width=1000,
    height=500
)




fig.show()

Плохо. Время после 2011-го года выкинем. Для наших исследований такое незначительное количество данных будет лишь портить и без того сложную статистику, не говоря уже о машинном обучении.

(там есть немного 2012 года, но лишь несколько месяцев)


Это противоречит идеям решения задачи от заказчика. А именно - четвертый пункт:

"Для моделирования ограничиться данными за 2012 год — они самые свежие."

2012 год может и самый свежий, но он НЕ полный.

Дальнейшее следование идеям сомнительно. Заказчик не знает о состоянии своих данных.

но мы проверим 2012 не только на ДТП. В обучении модели по заданию мне нужно использовать ИМЕННО 2012 год.

In [ ]:

query = '''
SELECT EXTRACT(MONTH FROM collision_date)::int AS month_num,
       COUNT(case_id) AS cases_count
FROM collisions
WHERE EXTRACT(YEAR FROM collision_date) <= 2011 
GROUP BY EXTRACT(MONTH FROM collision_date)
ORDER BY cases_count DESC;
'''


fig = px.bar(
    pd.read_sql_query(query, con=engine),
    
    y='cases_count',
    text='cases_count',
    title='Количество ДТП в месяц за 3 года',
    labels={'cases_count': 'Количество ДТП'},
    width=800,
    height=500
)



fig.show()

Почти ровное распределение. Значит и среднее будет таким же





In [ ]:
query = '''
WITH col AS
     (SELECT DATE_TRUNC('month', collision_date) AS date,
             COUNT(case_id) AS count_collisions
      FROM collisions
      WHERE EXTRACT(YEAR FROM collision_date) <= 2011 
      GROUP BY date)

SELECT EXTRACT(MONTH FROM date)::int AS month,
       AVG(count_collisions) AS count_collisions_mean
FROM col
GROUP BY month;
'''


fig = px.bar(
    pd.read_sql_query(query, con=engine),
    x='month',
    y='count_collisions_mean',
    text='count_collisions_mean',
    title='Среднее количество ДТП по месяцам (2009 - 2011)',
    labels={'count_collisions_mean': 'Среднее количество ДТП'},
    width=900,
    height=500
)


fig.show()

Наибольшее количество происшествий наблюдается в октябре месяце, затем следуют декабрь и март. 

Наибольшее количество ДТП случается в весенний и осенний периоды, также одним из лидеров по количеству ДТП является декабрь. Найденные закономерности могут быть связаны, например, с увеличением количества клиентов в указанные периоды, либо с изменением погодных условий.

Погода? Ну...

Скоро состоится первое совещание вашей рабочей группы. Чтобы обсуждение было конструктивным, каждый сотрудник должен понимать данные. Для этого вы должны создать подходящие аналитические задачи и поручить их решение коллегам. 


Примеры задач:
Проведите анализ серьёзности повреждений транспортного средства, исходя из состояния дороги в момент ДТП (связать collisions и parties);

Что это значит?

#### ОБРАЩЕНИЕ К КОЛЛЕГАМ

Вот как подключиться к базе данных. 

db_config = {
'user': 'praktikum_student', # имя пользователя,
'pwd': 'Sdf4$2;d-d30pp', # пароль,
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # порт подключения,
'db': 'data-science-vehicle-db' # название базы данных,
} 


1) Мне нужно исследовать связь между погодными условиями (weather_1 в collisions) и степенью повреждений (collision_damage в collisions) А то мое исследование среднее количество ДТП по месяцам (2009-2011 годы) дает основание предполагать вину погоды.

2) Определить процент виновных водителей в зависимости от их состояния (трезвый/нет). Еще на этапе осмотра я подумал об этом.

3) Как состояние дороги влияет на дтп?

Состояние дороги	ROAD_SURFACE	A — Dry (Сухая)
B — Wet (Мокрая)
C — Snowy or Icy (Заснеженная или обледенелая)
D — Slippery (Muddy, Oily, etc.) (Скользкая, грязная, маслянистая и т. д.)
- — Not Stated (Не указано)

4) Как освещение влияет  на повреждения при ДТП?


Освещение	LIGHTING	A — Daylight (Дневной свет)
B — Dusk-Dawn (Сумерки-Рассвет)
C — Dark-Street Lights (Темно-Уличные фонари)
D — Dark-No Street Lights (Темно-Нет уличных фонарей)
E — Dark-Street Lights Not Functioning (Темно-Уличные фонари не работают)
- — Not Stated (Не указано)


5) Зависимость состояния дороги и тяжести ДТП?

Дорожное состояние	ROAD_CONDITION_1	A — Holes, Deep Ruts (Ямы, глубокая колея)
B — Loose Material on Roadway (Сыпучий материал на проезжей части)
C — Obstruction on Roadway (Препятствие на проезжей части)
D — Construction or Repair Zone (Зона строительства или ремонта)
E — Reduced Roadway Width (Уменьшенная ширина проезжей части)
F — Flooded (Затоплено)
G — Other (Другое)
H — No Unusual Condition (Нет ничего необычного)
- — Not Stated (Не указано)

6) Тип дороги. Как он влияет на повреждения?

Тип дороги	LOCATION_TYPE	H — Highway (Шоссе)
I — Intersection (Перекрёсток)
R — Ramp (or Collector) (Рампа)
- or blank — Not State Highway (Не указано)


Найдите самые частые причины ДТП (таблица parties).

2.1. Создайте не менее шести задач для коллег. Опирайтесь на примеры и таблицы. 

2.2. Пропишите порядок решения для двух задач из списка. 

Реализуйте его. 


Обязательное условие — решение этих задач должно включать связь не менее 2-х таблиц. Пример прописанного порядка:

Создайте sql-запрос;
Постройте график;
Сделайте вывод.

Я пропишу порядок решения внизу. Вместе с графиками.



In [ ]:
query = '''
SELECT 
    c.WEATHER_1 AS weather,
    c.collision_damage,
    COUNT(*) AS accident_count
FROM 
    collisions AS c
GROUP BY 
    c.WEATHER_1, c.collision_damage
'''

fig = px.histogram(
    pd.read_sql_query(query, con=engine),
    x='weather',
    y='accident_count',
    color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от погодных условий',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch'],
        'weather': ['A', 'B', 'C', 'D', 'E', 'F', 'G', '-'] 
    },
    labels={
        'weather': 'Погодные условия',
        'accident_count': 'Количество ДТП',
        'collision_damage': 'Тяжесть повреждений'
    }
)


fig.show()


Погода WEATHER_1
(A — Clear (Ясно), B — Cloudy (Облачно), C — Raining (Дождь), D — Snowing (Снегопад), E — Fog (Туман), F —, Other (Другое), G — Wind (Ветер), - — Not Stated (Не указано))


 Отсюда понятно лишь то, что дтп происходят чаще всего в ясную погоду.

Это не значит, что не надо ездить в ясную погоду. Ох, надо смотреть процентное соотношение.

In [ ]:
query = '''
WITH weather_stats AS (
    SELECT 
        c.WEATHER_1 AS weather,
        c.collision_damage,
        COUNT(*) AS accident_count,
        SUM(COUNT(*)) OVER (PARTITION BY c.WEATHER_1) AS total_per_weather
    FROM 
        collisions AS c
    WHERE 
        c.WEATHER_1 IS NOT NULL
    GROUP BY 
        c.WEATHER_1, c.collision_damage
)
SELECT 
    weather,
    collision_damage,
    accident_count,
    (accident_count * 100.0 / total_per_weather) AS percentage
FROM 
    weather_stats
ORDER BY 
    weather, collision_damage
'''
fig = px.bar(
    pd.read_sql_query(query, con=engine),
    x='weather',
    y='percentage',
    color='collision_damage',
    barmode='stack', 
    title='Распределение тяжести ДТП по погодным условиям (%)',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch'],
        'weather': ['A', 'B', 'C', 'D', 'E', 'F', 'G', '-'] 
    },
    labels={
        'weather': 'Погодные условия',
        'percentage': 'Доля ДТП (%)',
        'collision_damage': 'Тяжесть повреждений'
    },
)


fig.show()

Заснеженность и "другое" предзнаменуют малые повреждения в ущерб остральным.

In [ ]:
query = '''
SELECT 
    c.PRIMARY_COLLISION_FACTOR AS collision_factor,
    c.collision_damage,
    COUNT(*) AS accident_count
FROM 
    collisions AS c
WHERE 
    c.PRIMARY_COLLISION_FACTOR != '-'
GROUP BY 
    c.PRIMARY_COLLISION_FACTOR, c.collision_damage
'''

fig = px.histogram(
    pd.read_sql_query(query, con=engine),
    x='collision_factor',
    y='accident_count',
    color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от основного фактора аварии',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch'],
        'collision_factor': ['A', 'B', 'C', 'D', 'E']  # Порядок факторов
    },
    labels={
        'collision_factor': 'Основной фактор аварии',
        'accident_count': 'Количество ДТП',
        'collision_damage': 'Тяжесть повреждений'
    }
)

fig.show()

Основной фактор аварии	PRIMARY_COLLISION_FACTOR

A — Code Violation (Нарушение правил ПДД)

B — Other Improper Driving (Другое неправильное вождение)

C — Other Than Driver (Кроме водителя)

D — Unknown (Неизвестно)

E — Fell Asleep (Заснул)

- — Not Stated (Не указано)

Из этого выводов не сделать. 

In [ ]:
query = '''
SELECT c.collision_damage,
       p.party_drug_physical
  FROM collisions AS c
       JOIN parties AS p ON c.case_id = p.case_id
 WHERE p.party_drug_physical IN ('under drug influence', 'sleepy/fatigued', 'impairment - physical');
'''

fig = px.histogram(
    pd.read_sql_query(query, con=engine), x='party_drug_physical', color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от состояния водителя',
    histfunc='count',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch']
    }
)

fig.show()


In [ ]:
fig = px.histogram(
    pd.read_sql_query(query, con=engine), x='party_drug_physical', color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от физического состояния водителя (проценты)',
    histfunc='count',
    histnorm='percent',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch']
    }
)

fig.show()

Состояние участника: физическое или с учётом принятых лекарств	PARTY_DRUG_PHYSICAL	

E — Under Drug Influence (Под воздействием лекарств)

F — Impairment — Physical (Ухудшение состояния)

G — Impairment Unknown (Не известно)

H — Not Applicable (Не оценивался)

I — Sleepy/Fatigued (Сонный/Усталый)

- — Not Stated (Не указано)



Рассмотрим Not Applicable:

In [ ]:
query = '''
SELECT c.collision_damage,
       p.party_drug_physical
  FROM collisions AS c
       JOIN parties AS p ON c.case_id = p.case_id
 WHERE p.party_drug_physical IN ('not applicable');
'''
fig = px.histogram(
    pd.read_sql_query(query, con=engine), x='party_drug_physical', color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от физического состояния водителя',
    histfunc='count',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch']
    }
)
fig.show()

Факторы "Под воздействием лекарств" и "Сонный/Усталый" явно сильно влияют на вероятность аварии.

Нельзя не отметить первый фактор и очень сильно выбивающийся процент fatal (Не подлежит восстановлению).

In [ ]:

query = '''
SELECT 
    county_location,
    COUNT(case_id) as accident_count
FROM collisions
GROUP BY county_location
ORDER BY accident_count DESC;
'''

# Создаем график
fig = px.bar(
    pd.read_sql_query(query, con=engine),
    x='county_location',
    y='accident_count',
    text='accident_count',
    title='<b>Распределение ДТП по округам Калифорнии</b>',
    labels={'accident_count': 'Количество ДТП', 'county_location': 'Округ'},
    width=1200,
    height=700,
    color='accident_count',
    color_continuous_scale='tempo'
)




fig.show()

Почему? Лос-Анжелес абсолютный лидер.

Но это лишь из-за населенности. Системе это нужно, если машина будет заезжать в Лос-Анжелес? Сразу появится тройной множитель риска аварии?

Нужно этот момент рассмотреть в выводе.

In [ ]:
query = '''
SELECT c.collision_damage,
       v.vehicle_age
  FROM collisions AS c
       JOIN vehicles AS v ON c.case_id = v.case_id;
'''

In [ ]:
fig = px.histogram(
    pd.read_sql_query(query, con=engine), x='vehicle_age', color='collision_damage',
    barmode='group',
    title='Зависимость тяжести ДТП от возраста ТС',
    histfunc='count',
    category_orders={
        'collision_damage': ['fatal', 'severe damage', 'middle damage', 'small damage', 'scratch']
    }
)
fig.show()


Ой, выбросы. Поставлю все машины старше 11 лет по одну категорию.

Но даже отсюда видно преобладание накоторых типов повреждений у трехлетних авто.

Тут очень легко ошибиться.

In [ ]:
query = '''
SELECT 
    CASE 
        WHEN v.vehicle_age BETWEEN 0 AND 1 THEN '0-1 года'
        WHEN v.vehicle_age BETWEEN 1 AND 2 THEN '1-2 года'
        WHEN v.vehicle_age BETWEEN 2 AND 3 THEN '2-3 года'
        WHEN v.vehicle_age BETWEEN 3 AND 4 THEN '3-4 года'
        WHEN v.vehicle_age BETWEEN 4 AND 5 THEN '4-5 года'
        WHEN v.vehicle_age BETWEEN 5 AND 6 THEN '5-6 года'
        WHEN v.vehicle_age BETWEEN 6 AND 7 THEN '6-7 года'
        WHEN v.vehicle_age BETWEEN 7 AND 8 THEN '7-8 года'
        WHEN v.vehicle_age BETWEEN 8 AND 9 THEN '8-9 года'
        WHEN v.vehicle_age BETWEEN 9 AND 10 THEN '9-10 года'
        WHEN v.vehicle_age BETWEEN 10 AND 11 THEN '10-11 года'
        WHEN v.vehicle_age BETWEEN 11 AND 12 THEN '11-12 лет'
        WHEN v.vehicle_age BETWEEN 12 AND 13 THEN '12-13 лет'
        ELSE '14+ лет'
    END AS vehicle_age_category,
    c.collision_damage,
    COUNT(*) as accident_count
FROM 
    collisions AS c
JOIN 
    vehicles AS v ON c.case_id = v.case_id
GROUP BY 
    vehicle_age_category, c.collision_damage
ORDER BY 
    vehicle_age_category, c.collision_damage
'''

fig = px.histogram(
    pd.read_sql_query(query, con=engine),
    x="vehicle_age_category",
    y="accident_count",
    color="collision_damage",
    barmode="group",
    title="Зависимость тяжести ДТП от возраста ТС",
    category_orders={
        "collision_damage": ["fatal", "severe damage", "middle damage", "small damage", "scratch"],
        "vehicle_age_category": ['0-1 года', '1-2 года', '2-3 года', '3-4 года','4-5 года','5-6 года','6-7 года','7-8 года','8-9 года','10-11 года','11-12 лет','12-13 лет','14+ лет']
    },
    labels={
        "vehicle_age_category": "Возраст ТС",
        "accident_count": "Количество ДТП",
        "collision_damage": "Тяжесть повреждений"
    }
)

fig.show()

Надо потом выбрать случаи, когда ДТП привело к любым повреждениям транспортного средства, кроме типа SCRATCH (царапина).


Серьёзность происшествия	COLLISION_DAMAGE

1 — FATAL ТС (Не подлежит восстановлению)

2 — SEVERE DAMAGE (Серьёзный ремонт, большая часть под замену/Серьёзное повреждение капитального строения)

3 — MIDDLE DAMAGE (Средний ремонт, машина в целом на ходу/Строение в целом устояло)

4 — SMALL DAMAGE (Отдельный элемент кузова под замену/покраску)

0 – SCRATCH (Царапина)

Можно сделать вывод, что тяжесть повреждений обратно пропорцональна их количествам.

За исключением царапин.

Почему трехлетние машины чаще попадают в дтп?
Возраст это не постоянная. Логично было бы тут видеть постепенное понижение графика. 
Скорее всего в этом возрасте у машины появляется какая-то скидка. И по этой причине автомобиль приобретают(арендуют) люди всех сортов. Это увеличивает аварии.
Возможно..

In [ ]:
query = '''
SELECT DISTINCT v.vehicle_type,
                COUNT(*) AS cnt_collisions
FROM collisions AS c
JOIN vehicles AS v ON c.case_id = v.case_id
GROUP BY v.vehicle_type
ORDER BY cnt_collisions
                 
        
'''



# Создаем график
fig = px.bar(
    pd.read_sql_query(query, con=engine),
    x='vehicle_type',
    y='cnt_collisions',
    text='cnt_collisions',
    title='<b>Распределение ДТП по типам транспортных средств</b>',
    labels={'cnt_collisions': 'Количество ДТП', },
    width=800,
    height=500,
    color='cnt_collisions',
    color_continuous_scale='Peach'
)



fig.show()

Тип кузова	VEHICLE_TYPE

MINIVAN
COUPE
SEDAN
HATCHBACK
OTHER

Антиреклама Седана и Купе. Может эти типы кузова самые распространенные?
Узнать это из датасета не получится, ведь тут только данные о ДТП.

**Выводы:**  

1. **Месяца и город:**  
   - Пик происшествий – октябрь, затем декабрь и март.  
   - Больше всего ДТП зафиксировано в Лос-Анджелесе.  

2. **Серьезность повреждений:**  
   - Основная тенденция: мелкие повреждения > царапины > средние > серьезные > фатальные.  


3. **Тип кузова:**  
   - Больше всего ДТП у **седанов**, затем **купе**.  
   - **Купе** лидирует по фатальным, серьезным и средним повреждениям, а также царапинам.  
   - **Седаны** – единственные с незначительными повреждениями.  



Не были рассмотрены:

В collisions:

Идентификационный  Номер в базе данных

Расстояние от главной дороги (метры)

Количество участников

Номер географических районов, где произошло ДТП

Категория нарушения	

Является ли место происшествия перекрёстком

Направление движения

Дополнительные участники ДТП	

Направление движения

В Parties:

Идентификационный  номер в базе данных

Номер участника происшествия

Тип участника происшествия (просто отберу авто в следующем этапе)

Виновность участника

Сумма страховки

Наличие телефона в автомобиле (возможности разговаривать по громкой связи)


В Vehicles:

Индекс текущей таблицы

Идентификационный номер в базе данных

Номер участника происшествия

Тип КПП


## Создайте модель для оценки водительского риска

Мне надо создать датафрейм с нужными данными

In [ ]:
query = '''
WITH parties_vehicles AS (
    SELECT 
        v.vehicle_type,
        v.vehicle_transmission, 
        v.vehicle_age,
        p.case_id,
        p.insurance_premium,
        p.at_fault
    FROM parties AS p
    JOIN vehicles AS v ON p.case_id = v.case_id 
        AND p.party_number = v.party_number
    WHERE p.party_type = 'car'
)

SELECT 
    c.county_city_location,
    c.distance,
    c.direction,
    c.weather_1,
    c.location_type,
    c.road_surface,
    c.road_condition_1, 
    c.lighting, 
    c.control_device,
    EXTRACT(MONTH FROM c.collision_date)::int AS collision_month,
    EXTRACT(DOW FROM c.collision_date)::int AS collision_dow,
    EXTRACT(HOUR FROM c.collision_time)::int AS collision_hour,
    p_v.vehicle_type,
    p_v.vehicle_transmission, 
    p_v.vehicle_age,
    p_v.insurance_premium,
    p_v.at_fault
FROM collisions AS c
JOIN parties_vehicles AS p_v ON c.case_id = p_v.case_id
WHERE EXTRACT(YEAR FROM c.collision_date) = 2012
  AND c.collision_damage != 'scratch';
'''
df = pd.read_sql_query(query, con=engine)

In [ ]:
df.info() 

## Предобработка и анализ данных

Этот этап явно нужен.

In [ ]:
df['direction'] = df['direction'].fillna('unknown')
df['weather_1'] = df['weather_1'].fillna('unknown')
df['location_type'] = df['location_type'].fillna('unknown')
df['road_surface'] = df['road_surface'].fillna('unknown')
df['road_condition_1'] = df['road_condition_1'].fillna('normal')
df['lighting'] = df['lighting'].fillna('unknown')
df['control_device'] = df['control_device'].fillna('unknown')
df['vehicle_transmission'] = df['vehicle_transmission'].fillna('unknown')


In [ ]:
df['at_fault'] = df['at_fault'].astype('bool')  # вместо int64

In [ ]:
df.isna().sum()#пропуски


Эти пропуски я уберу после разделения данных на тестовую и тренировочную выборки.

In [ ]:
df.head()

In [ ]:
df.duplicated().sum()# явн дубликаты


In [ ]:
df.drop_duplicates(inplace=True, ignore_index=True)

На этом этапе не было выявлено явных проблем. 

### EDA

#### Анализ категориальных признаков

In [ ]:
df['county_city_location'].nunique()

Начнем с категориальных признаков

Городов много. Но выше выяснено, что Лос-Анджелес в Калифорнии наиболее частое место, где происходит ДТП. Это не значит, что остальные города надо игнорировать. Это просто означает что из 496 мест к Лос-Анджелесу нужно "особое отношение"

In [ ]:
print(df['direction'].value_counts())

Все в основном едут на север и юг. 
Но в целом я вижу равномерное распределение по направлениям, что ожидаемо для дорожных происшествий.



In [ ]:
print(df['weather_1'].value_counts())

Большинство аварий происходит в ясную погоду, что может быть связано с большим количеством машин в таких условиях.


In [ ]:
print(df['location_type'].value_counts())

Автострады наиболее опасная локация. 



In [ ]:
print(df['road_surface'].value_counts())

Сухое покрытие доминирует, но мокрое покрытие представляет повышенный риск относительно своей частоты. Ведь она сильнее отличается от снега, чем сухость отличается от неё.

(хотя, может на дорогах в принципе мало снега из-за того что в Калифорнии его стараются сразу убрать?)

In [ ]:
print(df['road_condition_1'].value_counts())

Большинство аварий происходит в нормальных дорожных условиях



In [ ]:
print(df['lighting'].value_counts())

Дневное время наиболее рискованно и ночное время без освещения тоже очень опасно.

In [ ]:
print(df['control_device'].value_counts())

Отсутствие этого девайса может сулить аварию.


In [ ]:
print(df['vehicle_type'].value_counts())

sedan доминирует в статистике аварий

In [ ]:
print(df['vehicle_transmission'].value_counts())

Ручных и автоматических типов авто почти поровну.



In [ ]:
print(df['at_fault'].value_counts())

Баланс между виновными и невиновными сторонами

#### Анализ числовых признаков

In [ ]:
df.describe()

1) Большинство аварий происходит на коротких дистанциях 8 км

2) пик аварий в весну

3) Пик аварий в среду

4) Час пик(обед?) представляет большой риск аварий 

5) Средневозрастные автомобили наиболее представлены в авариях

6. Страховые премии коррелируют с рисковыми профилями водителей


ОБЩИЙ ПОРТРЕТ ВИНОВНИКА ДТП:


Весной в среду, владелец 3-летнего седана на ручном управлении без контрольного девайса поехал на обед по автостраде. Была ясная погода, а дорога сухая и ровная. Ехал на север и ничего не предвещало беды. Но он не справился с управлением.

In [ ]:
df = df.reset_index(drop=True)


In [ ]:
selected_cols = [
    'at_fault', 'insurance_premium', 'distance', 'direction',  
    'location_type', 'road_surface',
    'lighting', 'control_device', 'vehicle_type', 
    'vehicle_transmission', 'vehicle_age'
]


interval_cols = ['insurance_premium', 'distance', 'vehicle_age']

phi = df[selected_cols].phik_matrix(interval_cols=interval_cols)

In [ ]:


mask = np.triu(np.ones_like(phi, dtype=bool))

plt.figure(figsize=(14, 12))
heatmap = sns.heatmap(
    phi,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
    mask=mask,  # Скрываем верхний треугольник
    annot_kws={"size": 8},
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)

# Выделяем сильные корреляции (например, |r| > 0.7)
strong_corr_threshold = 0.7
for i in range(len(phi.columns)):
    for j in range(i + 1, len(phi.columns)):  # Проверяем только нижний треугольник
        corr_value = phi.iloc[i, j]
        if abs(corr_value) > strong_corr_threshold:
            heatmap.add_patch(
                plt.Rectangle(
                    (j, i),
                    1,
                    1,
                    fill=False,
                    edgecolor="red",
                    lw=2,
                    alpha=0.7,
                )
            )

plt.title("корреляционная матрица", pad=20, fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

Что у ЗАКАЗЧИКА за странные идеи решения?

Неужели ключевой фактор - это виновность?
На этом модель для основного задания не построить.

Поясню:
Не важно знать виновен ты или нет. ДТП уже произошло. А цель - предотвратить его.

Почему ключевой признак это то, что устанавливается уже после того, что нам надо было избежать?
Я не понял.


## Шаг 4. Создайте модель для оценки водительского риска
Подготовьте набор данных на основе первичного предположения заказчика:
 





Выберите тип виновника — только машина (car).




Возьмите случаи, когда ДТП привело к любым значимым повреждениям автомобиля любого из участников — все, кроме типа SCRATCH (царапина).


Для моделирования возьмите данные только за 2012 год.

Подготовка исходной таблицы должна проводиться с помощью sql-запроса.

Проведите первичный отбор факторов, необходимых для модели.

Изучите описание факторов. Нужно отобрать те, которые могут влиять на вероятность ДТП. Будет хорошо, если вы аргументируете свой выбор. 

Пример:

columms =['party_type',     # Тип участника происшествия. Таблица parties
          'party_sobriety', # Уровень трезвости виновника (точно может влиять) Таблица parties
           ......
         ] 
         
Проведите статистическое исследование отобранных факторов.

По результату исследовательского анализа внесите корректировки, если они нужны. Сделайте вывод.

Если необходимо, категоризируйте исходные данные, проведите масштабирование.

Подготовьте обучающую и тестовую выборки.

In [ ]:
X = df.drop(['at_fault'], axis=1)
y = df['at_fault']

cat_cols = X.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    X[col] = X[col].astype('category')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)


In [ ]:
numeric_cols = ['vehicle_age', 'insurance_premium']
X_train[numeric_cols] = X_train[numeric_cols].fillna(X_train[numeric_cols].median())
X_test[numeric_cols] = X_test[numeric_cols].fillna(X_train[numeric_cols].median())

X_train['collision_hour'] = X_train['collision_hour'].fillna(X_train['collision_hour'].mode()[0])
X_test['collision_hour'] = X_test['collision_hour'].fillna(X_train['collision_hour'].mode()[0])

In [ ]:
X_train.info()

In [ ]:
X_test.info()

In [ ]:
categorical_cols = [
    'party_sobriety',  
    'party_drug_physical',       
    'cellphone_in_use',         
    'vehicle_type',              
    'vehicle_transmission',  
    'direction',             
    'intersection',      
    'weather_1',            
    'location_type',         
    'collision_damage',        
    'primary_collision_factor',
    'pcf_violation_category',    
    'type_of_collision',         
    'motor_vehicle_involved_with',
    'road_surface',          
    'road_condition_1',          
    'lighting',                
    'control_device'         
]

In [ ]:
numeric_cols = [
    'insurance_premium',        
    'vehicle_age',        
    'distance'    
]


Я перенес collision_hour в categorical_cols

## Шаг 5. Найдите лучшую модель
Смоделируйте не менее 3-х типов моделей с перебором гиперпараметров.
Выберите метрику для оценки модели, исходя из поставленной бизнесом задачи. Обоснуйте свой выбор.
Оформите вывод в виде сравнительной таблицы.



In [ ]:
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore', drop='first'), 
     ['vehicle_type', 'vehicle_transmission', 'direction',
      'weather_1', 'location_type', 'road_surface', 'road_condition_1',
      'lighting', 'control_device']),
    
    (StandardScaler(),
     ['insurance_premium', 'vehicle_age', 'distance']),
    
    remainder='drop')

Три модели:

LogisticRegression

RandomForest

LightGBM

Модели выбраны. Первая выбрана за скорость, вторая за устойчивость к переобучению, а третья хорошо работает с категориальными признаками(коих у меня много)


Метрика выбрана.

Пусть это будет Recall. 
Почему?

Задача от заказчика выявить максимум потенциально опасных водителей/ситуаций, даже если часть прогнозов окажется ложной. Так мы не выдадим им авто.

Последствия ошибок:

False Negative (пропуск виновного) → Нельзя (приведет к невыявленным рискам и новым ДТП).

False Positive (ложное обвинение) → Нормально.

In [ ]:
pipe_lr = make_pipeline(
    preprocessor,
    LogisticRegression(
        random_state=42,
        class_weight='balanced', 
        max_iter=1000, 
        n_jobs=-1 
    )
)

pipe_rf = make_pipeline(
    preprocessor,
    RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced',
        verbose=0
    )
)

pipe_gbm = make_pipeline(
    OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
    LGBMClassifier(
        objective='binary',
        metric='f1',
        n_jobs=-1,
        verbosity=-1,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    )
)


### LogisticRegression


In [ ]:
pipe_lr.fit(X_train, y_train)

accuracy = pipe_lr.score(X_test, y_test)
print(f"Accuracy модели: {accuracy:.3f}")

param_grid = {
    'logisticregression__C': [0.001, 0.01, 0.1, 1, 10], 
    'logisticregression__penalty': ['l1', 'l2'],
    'logisticregression__solver': ['liblinear', 'saga'] 
}



In [ ]:
param_grid = {
    'logisticregression__C': [0.001, 0.01],
    'logisticregression__penalty': ['l2'], 
    'logisticregression__solver': ['saga'] 
}


In [ ]:
gs_lr = GridSearchCV(
    pipe_lr,
    param_grid,
    scoring='f1',
    cv=5, 
    n_jobs=-1,
    verbose=1
)


gs_lr.fit(X_train, y_train)

In [ ]:
print("Лучшие параметры:", gs_lr.best_params_)

### RandomForestClassifier

param_dist = {
    'randomforestclassifier__n_estimators': np.arange(50, 201),
    'randomforestclassifier__max_depth': np.arange(5, 50),
    'randomforestclassifier__min_samples_split': [2, 5, 10],
}

In [ ]:
param_dist = {
    'randomforestclassifier__n_estimators': np.arange(165, 170),
    'randomforestclassifier__max_depth': np.arange(45, 50),
    'randomforestclassifier__min_samples_split': [10],
}

In [ ]:
rs_rf = RandomizedSearchCV(
    pipe_rf,
    param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    error_score='raise',
    random_state=RANDOM_STATE
)

In [ ]:
rs_rf.fit(X_train, y_train)



In [ ]:
print("Лучшие параметры:", rs_rf.best_params_)

### LightGBM

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    df[col] = df[col].fillna('unknown').astype('category')


param_grid = {
    'lgbmclassifier__max_depth': [-1, 5, 10, 15],
    'lgbmclassifier__num_leaves': [15, 31, 63],
    'lgbmclassifier__learning_rate': [0.01, 0.05, 0.1],
    'lgbmclassifier__min_child_samples': [10, 20],
    'lgbmclassifier__reg_alpha': [0, 0.1],
    'lgbmclassifier__reg_lambda': [0, 0.1]
}


In [ ]:
param_grid = {
    'lgbmclassifier__max_depth': [-1, 5],
    'lgbmclassifier__num_leaves': [15],
    'lgbmclassifier__learning_rate': [0.1],
    'lgbmclassifier__min_child_samples': [10, 20],
    'lgbmclassifier__reg_alpha': [0.1],
    'lgbmclassifier__reg_lambda': [0]
}


In [ ]:
gs_gbm = GridSearchCV(
    pipe_gbm,
    param_grid,
    scoring='f1',
    cv=5, 
    n_jobs=-1,
    verbose=1,
    refit=True
)



In [ ]:
gs_gbm.fit(X_train, y_train)


In [ ]:
print("Лучшие параметры:", gs_gbm.best_params_)


Лучшие параметры: {'logisticregression__C': 0.001, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}


Лучшие параметры: {'randomforestclassifier__n_estimators': 167, 'randomforestclassifier__min_samples_split': 10, 'randomforestclassifier__max_depth': 45}


Лучшие параметры: {'lgbmclassifier__learning_rate': 0.1, 'lgbmclassifier__max_depth': -1, 'lgbmclassifier__min_child_samples': 10, 'lgbmclassifier__num_leaves': 15, 'lgbmclassifier__reg_alpha': 0.1, 'lgbmclassifier__reg_lambda': 0}

### Промежуточный вывод

In [ ]:
cv_results = {
    "LR": gs_lr.cv_results_['mean_test_score'].max(),
    "RF": rs_rf.cv_results_['mean_test_score'].max(),
    "LGBM": gs_gbm.cv_results_['mean_test_score'].max()
}


In [ ]:
best_model_name = max(cv_results, key=cv_results.get)

In [ ]:
if best_model_name == "LR":
    final_model = gs_lr.best_estimator_
elif best_model_name == "RF":
    final_model = rs_rf.best_estimator_
else:
    final_model = gs_gbm.best_estimator_

# ТЕСТ
y_test_pred = final_model.predict(X_test)
test_recall = recall_score(y_test, y_test_pred)

In [ ]:
print(f"Лучшая модель: {best_model_name}")
print(f"Recall на кросс-валидации: {cv_results[best_model_name]:.3f}")
print(f"Recall на тесте: {test_recall:.3f}\n")


Выбор пал на LGBM.

Общая точность (accuracy) примерно 0.62.

Модель слишком "осторожничает" с положительным классом (много False Negative)
Это хорошо, потому что другой результат при обратной ситуации мог бы привести к ДТП. А такая ошибка - допустима.

## Шаг 6. Проверьте лучшую модель в работе
Проведите графический анализ «Матрица ошибок». Выведите полноту и точность на график.

In [ ]:
prediction_gbm = gs_gbm.predict(X_test)


plt.figure(figsize=(15, 6))
cm = confusion_matrix(y_test, prediction_gbm, labels=gs_gbm.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=gs_gbm.classes_)
disp.plot(cmap='Blues', ax=plt.subplot(1, 2, 1))
plt.title('Матрица ошибок GradientBoosting')

precision, recall, _ = precision_recall_curve(y_test, gs_gbm.predict_proba(X_test)[:, 1])
PrecisionRecallDisplay(precision=precision, recall=recall).plot(ax=plt.subplot(1, 2, 2))
plt.title('Кривая точности-полноты (PR-кривая)')

plt.tight_layout()
plt.show()

print("\nОценка модели GradientBoosting:")
print(classification_report(y_test, prediction_gbm, digits=3))

Модель демонстрирует не очень приемлемое качество предсказаний для обоих классов и показывает недостаточную эффективность в определении положительного класса. Но она работает.
Для решения задачи критически важно минимизировать количество ложноотрицательных случаев. Кажется в этом случае, можно увеличить ложноположительные результаты. Специфика задачи такая.

1-4) Матрица ошибок (Confusion Matrix)

1) True Positive = 3257
То, для чего система создается. Система нашла 3257 реальных виновных водителей. Каждый такой человек позволяет применить меры.

2) False Negative = 2416 
Это худший сценарий. Мы пропустили 2416 виновных водителей. Они остались невыявленными, что может привести к нарушениям. 

3) False Positive = 1631
1631 невиновному водителю будет приписана вина.

4) True Negative = 3940
Это случаи успешного отсева. Мы корректно идентифицировали 3940 невиновных водителей.


5) Из всех случаев, когда модель сказала «виновен», только 66.6% оказались действительно виновными.
Когда система выдает сигнал о виновности водителя, она права в 67 случаях из 100 (примерно 2 из 3). Остальные 33 сигнала — это ложные тревоги, которые потребуют иных мер. 

Но лучше получить больше ложных тревог, чем пропустить реальную проблему.

6) Полнота
Модель смогла найти и correctly идентифицировать только 57.4% от всех реально виновных водителей.
Система на подобных данных обнаруживает меньше двух третей всех реальных нарушителей. Она пропускает почти половину (43%) виновных водителей. 


## Проведите анализ важности факторов ДТП

Проанализируйте важность основных факторов, влияющих на вероятность ДТП.


Погодные условия, состояние дороги, тип трансмиссии. Их влияние прослеживалось с момента анализа корреляций.

В итоге они не были исключены, 

In [ ]:
feature_names = preprocessor.get_feature_names_out()

coefficients = gs_lr.best_estimator_.named_steps['logisticregression'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nТоп-10 самых важных признаков:")
print(coef_df.head(10))

Выделяются:

1)Тип автомобиля - седан - Владельцы седанов реже попадают в аварии

2)Сontrol_device. Либо функционирует, либо нет. Это увеличивает риск на 24%. Отсутствие этих устройств повышает риск.

3)Страховая премия - Чем выше страховые премии - тем ниже риск. 
Более дорогие автомобили имеют лучшее оснащение безопасности?




### Дополнительное исследование
Для одного из выявленных важных факторов проведите дополнительное исследование:

Покажите график зависимости фактора и целевой переменной.

Предложите, чем можно оборудовать автомобиль, чтобы учесть этот фактор во время посадки водителя.

Для этого я исследую "Тип автомобиля" и его воздействие.
VEHICLE_TYPE	

In [ ]:
#тогда я создам отдельный датафрейм с главенствующими признаками
analysis_df = X_train[['vehicle_type']].copy()
analysis_df['vault'] = y_train.values 
vehicle_effect = analysis_df.groupby('vehicle_type')['vault'].mean().sort_values(ascending=False)

vehicle_labels = [vehicle_translation.get(label, label) for label in vehicle_effect.index]

plt.figure(figsize=(12, 8))
bars = plt.barh(vehicle_labels, vehicle_effect.values)

plt.title('Вероятность тяжелого ДТП в зависимости от типа кузова автомобиля', fontsize=14, pad=20)
plt.xlabel('Доля ДТП с пострадавшими', fontsize=12)
plt.ylabel('Тип кузова', fontsize=12)
plt.xlim(0, 1)

for i, v in enumerate(vehicle_effect.values):
    plt.text(v + 0.01, i, f'{v:.1%}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()
print("\nРейтинг по опасности (вероятность тяжелого ДТП):")
print(analysis_df['vault'].mean())


Напоминаю

Тип кузова	VEHICLE_TYPE	MINIVAN
COUPE
SEDAN
HATCHBACK
OTHER

1) Купе - самый опасный тип (61.7%)
На 38% опаснее седана
Каждое 6-е ДТП с купе приводит к тяжелым последствиям
Может дело в способе вождения, более низкой устойчивости? 

2. Седан - самый безопасный тип (44.6%)
На 28% безопаснее купе
Наиболее распространенный и предсказуемый в управлении
Хорошая пассивная безопасность

3. "Другие" типы (50.0%)
Промежуточный уровень риска





## Шаг 7. Сделайте общий вывод по модели


Кратко опишите лучшую модель.


Сделайте вывод: насколько возможно создание адекватной системы оценки риска при выдаче авто?


Какие факторы ещё необходимо собирать, чтобы улучшить модель?
Оформление: Выполните задание в Jupyter Notebook. Заполните программный код в ячейках типа code, текстовые пояснения — в ячейках типа markdown. Примените форматирование и заголовки.

## Выводы

Результаты моделирования:

Лучшая модель :LGBM.

Общая точность (accuracy) примерно 0.62.

Модель слишком "осторожничает" с положительным классом (много False Negative)
Это хорошо, потому что другой результат при обратной ситуации мог бы привести к ДТП. А такая ошибка - допустима.


Ключевые факторы риска:

Трезвость водителя и тип авто.

Надо было понять, помогут ли результаты моделирования и анализ важности факторов ответить на вопросы:

1) Возможно ли создать адекватную системы оценки водительского риска при выдаче авто?

Что значит адекватную? Создать систему на основании этих данных можно - но она будет не идеальной, а скорее будет как опциональная деталь. Надеятся на такую систему нельзя. Выдавать авто нужно людям изходя из самих людей, а не из статистики.
Но можно было бы создать систему на все авто, которая (каким-то образом), как только водитель забронирует автомобиль, сядет за руль и выберет маршрут, оценит уровень риска.

Если уровень риска высок, водитель увидит предупреждение и рекомендации по маршруту.

Высокий риск. Это какой? 10% аварии? 1%?

Нужно очень долго и много работать над подобными вопросами и тестировать все это.
На этот вопрос я ясного ответа не дам.

2) Какие ещё факторы нужно учесть?
Состояние водителя.

Еще надо бы добавить в данные несколько факторов:
Смертность. Однозначно.
Еще не помешал бы подробный тип нарушения пдд. Они бывают разные.

3) Нужно ли оборудовать автомобиль какими-либо датчиками или камерой?

Конечно. Для фиксирования ДТП хотя бы. Датчики погоды и другие, способные зафиксировать ключевые факторы риска. Они лишними не будут. Иначе как система будет оценивать уровень риска? Только предупредить об этом водителя надо.

Для ночного времени (23:00–4:00) — можно сделать будильник. К примеру поднять освещение в салоне так, что бы это не мешало водителю, но и не переводило его организм в режим сна.


Итог:
Модель эффективно предсказывает виновность в ДТП на основе ключевых факторов, но для оценки общего риска аварий требуется расширение данных. Внедрение предложенных мер может снизить частоту ДТП.


Я знаю, что автомобильные аварии - бич современного мира. В мегаполисах это одна из основных причин смерти и травм людей. А это данные содержат только информацию об участниках ДТП. Не видно общей картины.